### Designed by Dev Patel

In [15]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, random_split, Dataset
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve

In [16]:
base_image_directory = "/red/ruogu.fang/share/UKB/data/Eye/"
base_csv_path = "/blue/ruogu.fang/pateld3/combined_selected_images.csv"

df = pd.read_csv(base_csv_path)
df['image_path'] = df.apply(
    lambda row: f"{base_image_directory}/{row['image']}/{row['subject_id']}_{row['image'][:5]}_0_0.png",
    axis=1
)

df = df.drop(columns=['image'])

# Shuffle the dataframe
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.head(5)

,subject_id,disease,image_path
0,2174634,1,/red/ruogu.fang/share/UKB/data/Eye//21016_fund...
1,2294329,0,/red/ruogu.fang/share/UKB/data/Eye//21015_fund...
2,5435256,0,/red/ruogu.fang/share/UKB/data/Eye//21015_fund...
3,5107618,1,/red/ruogu.fang/share/UKB/data/Eye//21015_fund...
4,5786592,1,/red/ruogu.fang/share/UKB/data/Eye//21016_fund...


In [17]:
class UKBImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        label = self.df.iloc[idx]['disease']
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

In [18]:
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_df, val_df = train_test_split(df, test_size=0.2)

train_dataset = UKBImageDataset(train_df, transform=transform)
val_dataset = UKBImageDataset(val_df, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=1)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=1)

In [19]:
class InceptionV3Binary(nn.Module):
    def __init__(self):
        super(InceptionV3Binary, self).__init__()
        # Load pretrained InceptionV3 with default params (aux_logits=True)
        self.base_model = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)
        # Replace the final FC layer
        self.base_model.fc = nn.Identity()
        # New classifier for binary output
        self.classifier = nn.Linear(2048, 1)  # Binary classification (no sigmoid)
    
    def forward(self, x):
        # Get output from base model
        if self.training:
            # During training, inception returns a tuple (output, aux_output)
            output, _ = self.base_model(x)
        else:
            # During evaluation, it returns just the output
            output = self.base_model(x)
            
        # Flatten and classify
        output = torch.flatten(output, 1)
        output = self.classifier(output)
        return output

In [20]:
model = InceptionV3Binary()
# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [21]:
# 5. Training Setup
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

In [22]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Average Loss: {running_loss/len(train_loader):.4f}")

# Validation
model.eval()
y_pred, y_true = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)
        outputs = model(inputs)
        probs = torch.sigmoid(outputs).squeeze(1).cpu().numpy()
        y_pred.extend(probs)
        y_true.extend(labels.squeeze(1).cpu().numpy())
auc = roc_auc_score(y_true, y_pred)
print("AUC:", auc)

Epoch [1/20], Average Loss: 0.6651
Epoch [2/20], Average Loss: 0.5436
Epoch [3/20], Average Loss: 0.4892
Epoch [4/20], Average Loss: 0.4288
Epoch [5/20], Average Loss: 0.3039
Epoch [6/20], Average Loss: 0.2869
Epoch [7/20], Average Loss: 0.2224
Epoch [8/20], Average Loss: 0.2347
Epoch [9/20], Average Loss: 0.2162
Epoch [10/20], Average Loss: 0.1900
Epoch [11/20], Average Loss: 0.2546
Epoch [12/20], Average Loss: 0.1527
Epoch [13/20], Average Loss: 0.1017
Epoch [14/20], Average Loss: 0.1317
Epoch [15/20], Average Loss: 0.0873
Epoch [16/20], Average Loss: 0.1010
Epoch [17/20], Average Loss: 0.1587
Epoch [18/20], Average Loss: 0.1051
Epoch [19/20], Average Loss: 0.0986
Epoch [20/20], Average Loss: 0.0868
AUC: 0.7446808510638298


In [23]:
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_true, y_pred)

# You can find best threshold via Youden's J statistic
j_scores = tpr - fpr
best_threshold = thresholds[np.argmax(j_scores)]
print(f"Best threshold: {best_threshold:.2f}")

Best threshold: 0.55


In [24]:
from sklearn.metrics import f1_score, accuracy_score

# Threshold predictions
binary_preds = (np.array(y_pred) > 0.5).astype(int)

# Compute F1 Score
f1 = f1_score(y_true, binary_preds)
print(f"F1 Score: {f1:.4f}")

accuracy = accuracy_score(y_true, binary_preds)
print("Accuracy:", accuracy)

F1 Score: 0.7292
Accuracy: 0.717391304347826


In [25]:
import numpy as np
np.unique(binary_preds, return_counts=True)

(array([0, 1]), array([41, 51]))

In [26]:
torch.save(model.state_dict(), "inceptionv3_ukb_qc.pth")